In [ ]:
import os
from os.path import expanduser
home = expanduser("~/")

import sys
# sys.path.insert(0, '/global/u2/x/xshuang/gigalens-xh-dev/src')

# import sys
conda_env = sys.path[1]
del sys.path[1]

import os
# sys.path.append(f'{os.environ['HOME']}/gigalens_personal/gigalens/src')
sys.path.append(home+'/gigalens'+'/src')
sys.path.append(conda_env)
sys.path.append(home+'/GIGALens-Code/')
print(sys.path)



srcdir = os.path.join(home, "gigalens/src/")


In [ ]:
import tensorflow_probability.substrates.jax as tfp

from gigalens.jax.model import ForwardProbModel, BackwardProbModel
from gigalens.model import PhysicalModel
from gigalens.jax.simulator import LensSimulator
from gigalens.simulator import SimulatorConfig
from gigalens.jax.profiles.light import sersic
from gigalens.jax.profiles.mass import epl, shear, tnfw, tnfw_ellipse

import jax
from jax import random
import numpy as np
import optax
from jax import numpy as jnp
from matplotlib import pyplot as plt
import optax
import corner
import yaml
import pickle
from helpers import *
import blackjax
import importlib
from mclmc_alt import MCLMC
tfd = tfp.distributions

In [ ]:

import gigalens
importlib.reload(gigalens)
from gigalens.jax.profiles.mass import tnfw_ellipse
import lenstronomy
from lenstronomy.Data.pixel_grid import PixelGrid
import functools
from typing import List, Dict

import jax
import jax.numpy as jnp
import numpy as np
from jax import jit
from jax import lax
from lenstronomy.Util.kernel_util import subgrid_kernel
from objax.constants import ConvPadding
from objax.functional import average_pool_2d
from gigalens.jax.simulator import LensSimulator
import gigalens.model
import gigalens.simulator
from helpers import *

In [ ]:


import numpy as np
from scipy.optimize import brentq
from astropy.cosmology import FlatLambdaCDM
from lenstronomy.Cosmo.lens_cosmo import LensCosmo
from lenstronomy.LensModel.Profiles.nfw import NFW

def get_nfw_einstein_radius_lenstronomy(M200, c, z_lens, z_source, H0=70, Om0=0.3):
    """
    Calculates the NFW Einstein radius using lenstronomy components.
    
    Parameters:
    M200 : float - Halo mass (M_200) in M_sun.
    c    : float - Concentration parameter.
    z_lens : float - Redshift of the lens.
    z_source : float - Redshift of the source.
    """
    # 1. Initialize Cosmology and LensCosmo
    cosmo = FlatLambdaCDM(H0=H0, Om0=Om0)
    lens_cosmo = LensCosmo(z_lens=z_lens, z_source=z_source, cosmo=cosmo)
    
    # 2. Convert Physical NFW (M, c) to Angular NFW (Rs_angle, alpha_Rs)
    # Rs_angle: scale radius in arcseconds
    # alpha_Rs: deflection angle at the scale radius
    Rs_angle, alpha_Rs = lens_cosmo.nfw_physical2angle(M=M200, c=c)
    
    # 3. Define the NFW profile instance
    nfw_profile = NFW()

    # 4. Solve the lens equation: alpha(theta) - theta = 0
    # Note: For a spherical profile, alpha is only in the radial direction.
    def lens_equation(theta):
        # We calculate the deflection angle at radius 'theta'
        # derivatives() returns (f_x, f_y) - we just need the x-component for 1D
        alpha_theta, _ = nfw_profile.derivatives(x=theta, y=0, Rs=Rs_angle, alpha_Rs=alpha_Rs)
        return alpha_theta - theta

    try:
        # Search for a root (Einstein Radius) between a tiny radius and a large one
        # Typical cluster theta_E is < 100", so we search up to 50 * Rs_angle
        theta_E = brentq(lens_equation, 1e-9, Rs_angle * 50)
        return theta_E
    except (ValueError, RuntimeError):
        # Returns 0 if no solution exists (halo too diffuse to be critical)
        return 0.0


# M = 1e10 #1e10
# c = 10.0 # 10.0
# z_lens = 0.3
# z_source = 1.0

# cosmo = FlatLambdaCDM(H0=70, Om0=0.3)
# lens_cos = LensCosmo(z_lens=z_lens, z_source=z_source, cosmo=cosmo)
# Rs_angle, alpha_Rs = lens_cos.nfw_physical2angle(M, c)
# print(f"Rs_angle: {Rs_angle}, alpha_Rs: {alpha_Rs}")

# theta_E = get_nfw_einstein_radius_lenstronomy(M, c, z_lens, z_source)
# print(f"Einstein Radius: {theta_E} arcsec")

In [ ]:
kernel = np.load(os.path.join(srcdir, 'gigalens/assets/psf.npy')).astype(jnp.float32)
sim_config = SimulatorConfig(delta_pix=0.03, num_pix=125, supersample=2, kernel=kernel)

# NFW

phys_model = PhysicalModel([tnfw_ellipse.TNFW_Ellipse(), shear.Shear()], [sersic.SersicEllipse(use_lstsq=False)], [sersic.SersicEllipse(use_lstsq=False)])
lens_sim = LensSimulator(phys_model, sim_config, bs=1)

In [ ]:
# CURRENTLY SHOWING WITH KERNEL2 w/ lenstronomy bc wanted to perform comparison
# use variable kernel to get original version, not too different though

lens_light_params = [{'R_sersic': 0.1, 'n_sersic': 2., 'e1': 0.1, 'e2': 0.2, 'center_x': 0., 'center_y': 0., 'Ie': 20}]
source_light_params = [{'R_sersic': 1.5, 'n_sersic': 1.5, 'e1': 0.05, 'e2': -0.15, 'center_x': 0.2, 'center_y': 0., 'Ie': 70.}
]
Rs_angle = 0.7
alpha_Rs = 0.2
lens_params = [{'Rs': Rs_angle, 'alpha_Rs': alpha_Rs, 'r_trunc': 10., 'center_x': 0., 'center_y': 0., 'e1':0.0, 'e2':0.0}, {'gamma1': 0.02, 'gamma2': 0.01}]

test_full = lens_sim.simulate((lens_params,[], source_light_params))
fig, ax = plt.subplots()
ax.imshow(test_full)
ax.invert_yaxis()

plt.show()

In [ ]:
nfw_profile = NFW()

def lens_equation(theta):
    # We calculate the deflection angle at radius 'theta'
    # derivatives() returns (f_x, f_y) - we just need the x-component for 1D
    alpha_theta, _ = nfw_profile.derivatives(x=theta, y=0, Rs=Rs_angle, alpha_Rs=alpha_Rs)
    return alpha_theta - theta

theta_E = brentq(lens_equation, 1e-9, Rs_angle * 50)
print(f"Einstein Radius: {theta_E:.3f}\"")

In [ ]:

cosmo = FlatLambdaCDM(H0=70, Om0=0.3)
lens_cos = LensCosmo(z_lens=z_lens, z_source=z_source, cosmo=cosmo)
rho0, Rs, c, r200, M = lens_cos.nfw_angle2physical(Rs_angle, alpha_Rs)
print(f"Mass : {M:.2e}, c: {c:.2f}")